# PRJEB30386 — проверка синтетических PE150-ридов всех цепей с помощью Bowtie2

Сгенерированные смешанные риды IGH+IGK+IGL выравниваются на точные фрагменты, использованные InSilicoSeq. Рассчитываются доли выровненных и корректно спаренных ридов, частота ошибок и совпадение с исходным фрагментом. Проверка оценивает симуляцию, но не восстанавливает биологические нуклеотиды, замещённые V-праймерами.


### 1. Проверка окружения

In [ ]:
import os,sys,sysconfig,subprocess,shutil
_ENV_CANDIDATES=[os.environ.get("BCR_ENV",""),"/data/user/epishkin/conda/envs/bcr_env","/opt/conda/envs/bcr_env","/Users/epishkin/mamba/envs/bcr_env"]
_CONDA_ENV=next((p for p in _ENV_CANDIDATES if p and os.path.isdir(p+"/bin")),None)
if not _CONDA_ENV: raise FileNotFoundError("bcr_env not found")
os.environ["PATH"]=_CONDA_ENV+"/bin:"+os.environ.get("PATH",""); os.environ["PYTHONNOUSERSITE"]="1"
for _site in [_CONDA_ENV+"/lib/python3.11/site-packages",_CONDA_ENV+"/lib/python3.12/site-packages",sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path: sys.path.insert(0,_site)
for tool in ("bowtie2","bowtie2-build","samtools"):
    p=shutil.which(tool)
    if not p: raise RuntimeError(f"{tool} missing")
    print(tool,"->",p)


### 2. Параметры

In [ ]:
from pathlib import Path
VOLUME=Path(os.environ.get("BCR_VOLUME","/data/user/epishkin"))
if not (VOLUME/"results/PRJEB30386").exists():
    found=None
    for start in (Path.cwd().resolve(),Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start,*start.parents):
            if (candidate/"results/PRJEB30386").exists() and (candidate/"scripts").is_dir(): found=candidate; break
        if found: break
    if not found: raise FileNotFoundError("Cannot locate repository or data root; set BCR_VOLUME")
    VOLUME=found
DATASET="PRJEB30386"; SAMPLES=["PRJEB30386_all_chains"]
SIM_VARIANTS=("insilicoseq_150bp_novaseq","insilicoseq_150bp_custom_umi_consensus")
SIM_VARIANT=os.environ.get("SIM_VARIANT",SIM_VARIANTS[0])
if SIM_VARIANT not in SIM_VARIANTS: raise ValueError(f"SIM_VARIANT must be one of {SIM_VARIANTS}")
SIM_DIR=VOLUME/"results"/DATASET/"simulated"/SIM_VARIANT; FASTQ_DIR=SIM_DIR/"06_fastq_pe150"; SHARED_TEMPLATES_DIR=SIM_DIR/"00_primary_truth"
ALIGN_DIR=SIM_DIR/"alignment"; INDEX_DIR=ALIGN_DIR/"index"; BAM_DIR=ALIGN_DIR/"bam"; LOGS_DIR=ALIGN_DIR/"logs"; QC_DIR=ALIGN_DIR/"qc"
for d in (INDEX_DIR,BAM_DIR,LOGS_DIR,QC_DIR): d.mkdir(parents=True,exist_ok=True)
NPROC=8; FORCE=False
print("SIM_DIR:",SIM_DIR); print("ALIGN_DIR:",ALIGN_DIR)


### 3. Индекс `bowtie2-build` для шаблонов каждого образца

Индекс строится по `{sample}_templates.fasta`: это те же последовательности, которые передаются через `--genomes` при генерации ридов, а не общий объединённый референс.


In [ ]:
import time

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(cmd)
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}); see {log_path}")
    print(f"done: elapsed={elapsed/60:.1f} min")


def build_index(sample, force=FORCE):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_fragments.fasta"
    if not templates_fa.exists():
        raise FileNotFoundError(f"Missing templates for {sample}: {templates_fa}")
    index_prefix = INDEX_DIR / sample
    done_marker = Path(str(index_prefix) + ".1.bt2")
    if done_marker.exists() and not force:
        print(f"[{sample}] [skip] index exists: {index_prefix}")
        return index_prefix
    run_with_heartbeat(
        ["bowtie2-build", "--threads", str(NPROC), str(templates_fa), str(index_prefix)],
        LOGS_DIR / f"{sample}_bowtie2_build.log",
    )
    return index_prefix


In [ ]:
indexes = {sample: build_index(sample) for sample in SAMPLES}


### 4. Локальное выравнивание симулированных R1/R2 на собственный референс

Режим `--local` учитывает ошибки и инделы InSilicoSeq вблизи концов ридов; `--end-to-end` для этой модели не используется.


In [ ]:
def align_sample(sample,force=FORCE):
    bam_path=BAM_DIR/f"{sample}.bam"; sam_path=BAM_DIR/f"{sample}.sam"
    if bam_path.exists() and Path(str(bam_path)+".bai").exists() and not force: print(f"[{sample}] [skip] {bam_path}"); return bam_path
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()): raise FileNotFoundError(f"Missing synthetic mates: {r1} / {r2}")
    run_with_heartbeat(["bowtie2","--local","-p",str(NPROC),"-x",str(indexes[sample]),"-1",str(r1),"-2",str(r2),"-S",str(sam_path)],LOGS_DIR/f"{sample}_bowtie2_align.log")
    run_with_heartbeat(["samtools","sort","-@",str(NPROC),"-o",str(bam_path),str(sam_path)],LOGS_DIR/f"{sample}_samtools_sort.log")
    subprocess.run(["samtools","index",str(bam_path)],check=True); sam_path.unlink(missing_ok=True); return bam_path


In [ ]:
bams = {sample: align_sample(sample) for sample in SAMPLES}


### 5. Контроль качества: выравнивание, несовпадения и исходный шаблон

`samtools flagstat` вычисляет доли выровненных и корректно спаренных ридов. `samtools stats` оценивает частоту несовпадений на нуклеотид относительно референса. Доля совпадения с исходным шаблоном определяется по соответствию `RNAME` в BAM идентификатору шаблона, закодированному ISS в имени рида (`{sample}_u{idx}_n{count}_.../1`).


In [ ]:
import re
import pysam

FLAGSTAT_MAPPED_RE = re.compile(r"(\d+) \+ \d+ mapped \(([\d.]+)")
FLAGSTAT_PAIRED_RE = re.compile(r"(\d+) \+ \d+ properly paired \(([\d.]+)")
STATS_ERROR_RATE_RE = re.compile(r"^error rate:\s+([\d.]+)")


def parse_flagstat(bam_path):
    out = subprocess.run(["samtools", "flagstat", str(bam_path)], capture_output=True, text=True, check=True).stdout
    mapped = FLAGSTAT_MAPPED_RE.search(out)
    paired = FLAGSTAT_PAIRED_RE.search(out)
    return {
        "mapped_pct": float(mapped.group(2)) if mapped else None,
        "properly_paired_pct": float(paired.group(2)) if paired else None,
    }


def parse_error_rate(bam_path):
    out = subprocess.run(["samtools", "stats", str(bam_path)], capture_output=True, text=True, check=True).stdout
    for line in out.splitlines():
        m = STATS_ERROR_RATE_RE.match(line)
        if m:
            return float(m.group(1))
    return None


def template_id_from_read_name(qname):
    # Имя рида ISS: {template_id}_{i}_{j}/1; template_id оканчивается на _n<count>,
    # поэтому удаляются ровно два последних сегмента "_<int>" после удаления /1 или /2.
    parts = qname.split("_")
    return "_".join(parts[:-2])


def self_origin_rate(bam_path, sample, max_reads=200_000):
    total = 0
    matched = 0
    with pysam.AlignmentFile(str(bam_path), "rb") as bam:
        for read in bam.fetch(until_eof=True):
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue
            total += 1
            if total > max_reads:
                break
            expected = template_id_from_read_name(read.query_name)
            if expected == read.reference_name:
                matched += 1
    return (matched / total) if total else None


In [ ]:
import csv

qc_rows = []
for sample in SAMPLES:
    bam_path = bams[sample]
    row = {"sample": sample, "bam": str(bam_path)}
    row.update(parse_flagstat(bam_path))
    row["error_rate"] = parse_error_rate(bam_path)
    row["self_origin_rate"] = self_origin_rate(bam_path, sample)
    qc_rows.append(row)
    print(sample, row)

qc_path = QC_DIR / "alignment_qc.tsv"
with open(qc_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(qc_rows[0].keys()), delimiter="\t")
    writer.writeheader()
    writer.writerows(qc_rows)
print(f"wrote {qc_path}")


### Интерпретация

Референсом служит FASTA с фрагментами, переданный в `iss generate`. Высокие доли выровненных, корректно спаренных ридов и совпадений с исходным фрагментом подтверждают генерацию и связь с эталонными данными. Анализ выявляет ошибки симуляции и множественное выравнивание, но не определяет последовательность, скрытую исходными V-праймерами.
